# Credit Risk Decision Engine — Deployable Logistic Regression Baseline

This baseline uses only the explicit production contract. The untouched final test is reserved before any comparison and is not inspected here.


In [1]:
from pathlib import Path
import sys
import warnings

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, train_test_split

from src.components.data_loader import load_training_data
from src.components.data_preprocessor import build_preprocessor_for_frame, prepare_model_features, select_production_raw_features
from src.components.model_evaluator import evaluate_probabilities
from src.config import MODEL_SELECTION_FOLDS, RANDOM_STATE, TARGET_COLUMN, TEST_SIZE


## Reserve final test and generate fold-fitted development predictions


In [2]:
df = load_training_data()
raw = select_production_raw_features(df)
development_x, reserved_test_x, development_y, reserved_test_y = train_test_split(
    raw, df[TARGET_COLUMN], test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=df[TARGET_COLUMN]
)
features = prepare_model_features(development_x)
oof = np.zeros(len(features))
cv = StratifiedKFold(MODEL_SELECTION_FOLDS, shuffle=True, random_state=RANDOM_STATE)
for train_idx, validation_idx in cv.split(features, development_y):
    fold_train = features.iloc[train_idx]
    fold_validation = features.iloc[validation_idx]
    preprocessor = build_preprocessor_for_frame(fold_train, scale_numeric=True)
    train_matrix = preprocessor.fit_transform(fold_train)
    validation_matrix = preprocessor.transform(fold_validation)
    model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
    model.fit(train_matrix, development_y.iloc[train_idx])
    oof[validation_idx] = model.predict_proba(validation_matrix)[:, 1]
baseline_metrics = evaluate_probabilities(development_y, oof)
display(pd.Series({key: value for key, value in baseline_metrics.items() if key != "risk_deciles"}))
print("Reserved final-test rows not evaluated:", f"{len(reserved_test_x):,}")


threshold                                               0.5
accuracy                                           0.919247
precision                                             0.125
recall                                              0.00005
f1                                                 0.000101
roc_auc                                            0.639735
pr_auc                                             0.132497
brier_score                                        0.072781
confusion_matrix                  [[226141, 7], [19859, 1]]
top_10_percent_default_capture                     0.201208
dtype: object

Reserved final-test rows not evaluated: 61,503


A 0.50 classification cutoff is only a diagnostic and is unsuitable as an assumed credit policy threshold for this imbalanced target. Probability ranking, calibration, risk concentration, and an explicit review-capacity policy are evaluated separately.
